In [1]:
import glob
import numpy as np
import pandas as pd
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler


KYIV_FILE = "Kyiv.csv"
df_kyiv = pd.read_csv(KYIV_FILE)

ukraine_files = [
    f for f in glob.glob("*.csv")
    if f != KYIV_FILE and not f.endswith("_links.csv")
    and not f.endswith("_for_analysis.csv") and not f.endswith("_ML.csv")
]
print("Файли України:", ukraine_files)
df_ukraine = pd.concat([pd.read_csv(f) for f in ukraine_files], ignore_index=True)

print("Kyiv:", df_kyiv.shape)
print("Ukraine (без Києва):", df_ukraine.shape)


Файли України: ['Cherkasy.csv', 'Chernihiv.csv', 'Chernivtsi.csv', 'Dnipro.csv', 'Ivano-Frankivsk.csv', 'Kharkiv.csv', 'Kherson.csv', 'Khmelnytskyi.csv', 'Kropyvnytskyi.csv', 'Kyiv_preprocessed.csv', 'Lutsk.csv', 'Lviv.csv', 'Mykolaiv.csv', 'Odesa.csv', 'Poltava.csv', 'Rivne.csv', 'Sumy.csv', 'Ternopil.csv', 'Uzhhorod.csv', 'Vinnytsia.csv', 'Zaporizhzhia.csv', 'Zhytomyr.csv']
Kyiv: (11486, 39)
Ukraine (без Києва): (23580, 43)


In [2]:
df_ukraine.info()

<class 'pandas.DataFrame'>
RangeIndex: 23580 entries, 0 to 23579
Data columns (total 43 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   group_id               12622 non-null  float64
 1   has_duplicates         12622 non-null  object 
 2   url                    23580 non-null  str    
 3   city                   23580 non-null  str    
 4   district               23580 non-null  str    
 5   residential_complex    5644 non-null   str    
 6   lat                    12243 non-null  float64
 7   lon                    12243 non-null  float64
 8   distance_to_center_km  12243 non-null  float64
 9   poi_name               9462 non-null   object 
 10  poi_distance_m         9462 non-null   float64
 11  geo_region             23580 non-null  str    
 12  price                  23575 non-null  float64
 13  num_of_rooms           23579 non-null  float64
 14  area                   23553 non-null  float64
 15  living_area  

In [3]:
df_kyiv.info()

<class 'pandas.DataFrame'>
RangeIndex: 11486 entries, 0 to 11485
Data columns (total 39 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   group_id               11486 non-null  int64  
 1   has_duplicates         11486 non-null  bool   
 2   url                    11486 non-null  str    
 3   city                   11486 non-null  str    
 4   district               11486 non-null  str    
 5   residential_complex    6530 non-null   str    
 6   lat                    11478 non-null  float64
 7   lon                    11478 non-null  float64
 8   distance_to_center_km  11478 non-null  float64
 9   poi_name               2294 non-null   str    
 10  poi_distance_m         2294 non-null   float64
 11  geo_region             11486 non-null  str    
 12  price                  11486 non-null  int64  
 13  num_of_rooms           11486 non-null  int64  
 14  area                   11486 non-null  float64
 15  living_area  

In [4]:
df_ukraine.isna().sum()

group_id                 10958
has_duplicates           10958
url                          0
city                         0
district                     0
residential_complex      17936
lat                      11337
lon                      11337
distance_to_center_km    11337
poi_name                 14118
poi_distance_m           14118
geo_region                   0
price                        5
num_of_rooms                 1
area                        27
living_area               8546
kitchen_area               138
floor                        4
floors_in_house              4
house_type                   0
heating                      0
wall_type                    0
year_of_building          4749
ceiling_height           13737
freshly_renovated          192
bedroom_count            18188
balcony_count            19311
toilets_count            17896
kitchen_type             18338
hot_water_type           18445
has_gas                  14197
autonomy_power           20396
autonomy

In [5]:
df_kyiv.isna().sum()

group_id                    0
has_duplicates              0
url                         0
city                        0
district                    0
residential_complex      4956
lat                         8
lon                         8
distance_to_center_km       8
poi_name                 9192
poi_distance_m           9192
geo_region                  0
price                       0
num_of_rooms                0
area                        0
living_area              5178
kitchen_area               42
floor                       0
floors_in_house             0
house_type                  0
heating                     0
wall_type                   0
year_of_building          155
ceiling_height             80
freshly_renovated         123
bedroom_count            7545
balcony_count            8547
toilets_count            7425
kitchen_type             7707
hot_water_type           8083
has_gas                  2826
autonomy_power           8732
autonomy_heat            3391
autonomy_w

In [6]:
PRICE_SQM_MIN, PRICE_SQM_MAX = 100, 10000


def clean_price_and_floors(df: pd.DataFrame, name: str) -> pd.DataFrame:
    before = len(df)
    df = df[df["price"].notna() & df["price"].between(PRICE_SQM_MIN, PRICE_SQM_MAX)]
    df = df[df["area"].notna() & (df["area"] > 0)]
    df = df[
        df["floor"].isna() | df["floors_in_house"].isna()
        | ((df["floor"] >= 1) & (df["floor"] <= df["floors_in_house"]))
    ]
    df = df[df["lat"].notna() & df["lon"].notna() & df["distance_to_center_km"].notna()]
    print(f"{name}: прибрано {before - len(df)} рядків (ціна/площа/поверховість/координати). Лишилось: {len(df)}")
    return df


df_ukraine = clean_price_and_floors(df_ukraine, "Ukraine")
df_kyiv = clean_price_and_floors(df_kyiv, "Kyiv")

Ukraine: прибрано 11397 рядків (ціна/площа/поверховість/координати). Лишилось: 12183
Kyiv: прибрано 103 рядків (ціна/площа/поверховість/координати). Лишилось: 11383


In [7]:
def room_group(df: pd.DataFrame) -> pd.Series:
    return df["num_of_rooms"].clip(upper=4).fillna(-1).astype(int).astype(str)


def remove_outliers_iqr_grouped(df: pd.DataFrame, cols: list[str], k: float = 1.5) -> pd.DataFrame:
    groups = room_group(df)
    keep_mask = pd.Series(True, index=df.index)

    for group_value in pd.unique(groups):
        idx = df.index[groups == group_value]
        group_df = df.loc[idx]
        group_mask = pd.Series(True, index=idx)
        for col in cols:
            q1, q3 = group_df[col].quantile(0.25), group_df[col].quantile(0.75)
            iqr = q3 - q1
            lower, upper = q1 - k * iqr, q3 + k * iqr
            group_mask &= group_df[col].between(lower, upper)
            print(f"  [rooms={group_value}] {col}: lower={lower:.1f}, upper={upper:.1f}")
        keep_mask.loc[idx] = group_mask

    return df[keep_mask]


before = len(df_ukraine)
df_ukraine = remove_outliers_iqr_grouped(df_ukraine, ["price", "area"])
print(f"\nПрибрано {before - len(df_ukraine)} викидів по Україні. Лишилось: {len(df_ukraine)}")

  [rooms=2] price: lower=-264.4, upper=2520.6
  [rooms=2] area: lower=10.9, upper=105.4
  [rooms=3] price: lower=-288.5, upper=2355.5
  [rooms=3] area: lower=15.6, upper=146.0
  [rooms=1] price: lower=-449.5, upper=2930.5
  [rooms=1] area: lower=13.5, upper=65.5
  [rooms=4] price: lower=-493.0, upper=2775.0
  [rooms=4] area: lower=-26.8, upper=279.4

Прибрано 786 викидів по Україні. Лишилось: 11397


In [8]:
before = len(df_kyiv)
df_kyiv = remove_outliers_iqr_grouped(df_kyiv, ["price", "area"])
print(f"\nПрибрано {before - len(df_kyiv)} викидів в Києві. Лишилось: {len(df_kyiv)}")

  [rooms=3] price: lower=-509.5, upper=4078.5
  [rooms=3] area: lower=23.8, upper=156.9
  [rooms=2] price: lower=-229.0, upper=3779.0
  [rooms=2] area: lower=15.5, upper=110.7
  [rooms=1] price: lower=-10.0, upper=3750.0
  [rooms=1] area: lower=15.2, upper=67.7
  [rooms=4] price: lower=-1244.5, upper=6039.5
  [rooms=4] area: lower=10.7, upper=328.9

Прибрано 778 викидів в Києві. Лишилось: 10605


In [9]:
for df in (df_ukraine, df_kyiv):
    df["in_residential_complex"] = df["residential_complex"].notna().astype(int)
    df["has_poi"] = df["poi_name"].notna().astype(int)
    df.drop(columns=["residential_complex", "poi_name"], inplace=True)

In [10]:
NUMERIC_TARGETS = ["year_of_building", "ceiling_height", "living_area", "kitchen_area"]


def impute_numeric(df: pd.DataFrame, location_cols: list[str]) -> pd.DataFrame:
    location_onehot = pd.get_dummies(df[location_cols], columns=location_cols)

    numeric_cols = ["area", "num_of_rooms"] + NUMERIC_TARGETS
    numeric_source = df[numeric_cols].copy()
    numeric_source["num_of_rooms"] = numeric_source["num_of_rooms"].fillna(numeric_source["num_of_rooms"].median())

    scaler = StandardScaler()
    scaled_numeric = pd.DataFrame(
        scaler.fit_transform(numeric_source), columns=numeric_cols, index=df.index,
    )

    knn_input = pd.concat([scaled_numeric, location_onehot], axis=1)
    imputer = KNNImputer(n_neighbors=5, weights="distance")
    imputed_array = imputer.fit_transform(knn_input)
    imputed = pd.DataFrame(imputed_array, columns=knn_input.columns, index=df.index)

    imputed_numeric = pd.DataFrame(
        scaler.inverse_transform(imputed[numeric_cols]), columns=numeric_cols, index=df.index,
    )

    df["year_of_building"] = imputed_numeric["year_of_building"].round().astype("Int64")
    df["ceiling_height"] = imputed_numeric["ceiling_height"].round(1)
    df["living_area"] = imputed_numeric["living_area"].round(1)
    df["kitchen_area"] = imputed_numeric["kitchen_area"].round(1)
    return df


df_ukraine = impute_numeric(df_ukraine, location_cols=["district", "city"])
df_ukraine[NUMERIC_TARGETS].isna().sum()

year_of_building    0
ceiling_height      0
living_area         0
kitchen_area        0
dtype: int64

In [11]:
df_kyiv = impute_numeric(df_kyiv, location_cols=["district", "city"])
df_kyiv[NUMERIC_TARGETS].isna().sum()

year_of_building    0
ceiling_height      0
living_area         0
kitchen_area        0
dtype: int64

In [12]:
def check_area_consistency(df: pd.DataFrame, name: str) -> pd.DataFrame:
    before = len(df)
    df = df[df["living_area"] <= df["area"]]
    df = df[df["kitchen_area"] <= df["area"]]
    df = df[(df["living_area"] + df["kitchen_area"]) <= df["area"]]
    print(f"{name}: прибрано {before - len(df)} рядків (непослідовні площі). Лишилось: {len(df)}")
    return df


df_ukraine = check_area_consistency(df_ukraine, "Ukraine")
df_kyiv = check_area_consistency(df_kyiv, "Kyiv")

for df in (df_ukraine, df_kyiv):
    df["poi_distance_m"] = df["poi_distance_m"].fillna(df["poi_distance_m"].median())

Ukraine: прибрано 139 рядків (непослідовні площі). Лишилось: 11258
Kyiv: прибрано 131 рядків (непослідовні площі). Лишилось: 10474


In [13]:
BOOLEAN_TRISTATE_COLS = [
    "has_gas", "autonomy_power", "autonomy_heat", "autonomy_water",
    "autonomy_net", "autonomy_lift", "freshly_renovated",
]


def fill_boolean_tristate(df: pd.DataFrame) -> pd.DataFrame:
    for col in BOOLEAN_TRISTATE_COLS:
        df[f"{col}_unknown"] = df[col].isna().astype(int)
        mode_val = df[col].mode(dropna=True)
        fill_val = bool(mode_val.iloc[0]) if not mode_val.empty else False
        df[col] = df[col].fillna(fill_val).astype(int)
    return df


df_ukraine = fill_boolean_tristate(df_ukraine)
df_kyiv = fill_boolean_tristate(df_kyiv)

In [14]:
for df in (df_ukraine, df_kyiv):
    for col in ["house_type", "wall_type", "heating", "kitchen_type", "hot_water_type"]:
        df[col] = df[col].fillna("Unknown")

df_ukraine[["house_type", "wall_type", "heating", "kitchen_type", "hot_water_type"]].isna().sum()

house_type        0
wall_type         0
heating           0
kitchen_type      0
hot_water_type    0
dtype: int64

In [15]:
COUNT_COLS = ["bedroom_count", "toilets_count", "balcony_count"]

for df in (df_ukraine, df_kyiv):
    for col in COUNT_COLS:
        df[f"{col}_missing"] = df[col].isna().astype(int)
        df[col] = df[col].fillna(df[col].median())

df_ukraine[COUNT_COLS].isna().sum()

bedroom_count    0
toilets_count    0
balcony_count    0
dtype: int64

In [16]:
df_ukraine["price"] = df_ukraine["price"].round(0)
df_ukraine["area"] = df_ukraine["area"].round(1)
df_ukraine[["price", "area", "ceiling_height"]].describe()

,price,area,ceiling_height
count,11258.000000,11258.000000,11258.000000
mean,1132.556404,61.003002,2.738417
std,510.998405,28.686775,0.232741
min,154.000000,14.000000,2.400000
25%,756.000000,43.000000,2.600000
50%,1000.000000,54.200000,2.700000
75%,1437.000000,71.000000,2.800000
max,2922.000000,275.000000,5.000000


In [17]:
df_kyiv["price"] = df_kyiv["price"].round(0)
df_kyiv["area"] = df_kyiv["area"].round(1)
df_kyiv[["price", "area", "ceiling_height"]].describe()

,price,area,ceiling_height
count,10474.000000,10474.000000,10474.000000
mean,1841.784800,74.022704,2.783426
std,771.494684,41.798158,0.249082
min,336.000000,17.000000,2.000000
25%,1282.000000,45.900000,2.600000
50%,1648.000000,64.000000,2.700000
75%,2250.000000,88.000000,3.000000
max,6011.000000,328.900000,5.000000


In [18]:
df_kyiv.to_csv("Kyiv_flats_for_analysis.csv", index=False)
df_ukraine.to_csv("Ukraine_flats_for_analysis.csv", index=False)

In [19]:
before = len(df_ukraine)
df_encoded_ukraine = df_ukraine.drop_duplicates(subset="group_id", keep="first").copy()
print(f"Прибрано {before - len(df_encoded_ukraine)} дублікатів за group_id (Україна)")

df_encoded_ukraine = df_encoded_ukraine.drop(columns=["url", "group_id", "geo_region", "text"])

categorical_cols_ukraine = ["house_type", "wall_type", "heating", "kitchen_type", "hot_water_type", "district", "city"]
df_encoded_ukraine = pd.get_dummies(df_encoded_ukraine, columns=categorical_cols_ukraine, drop_first=True)
df_encoded_ukraine.to_csv("Ukraine_flats_ML.csv", index=False)
df_encoded_ukraine.shape

Прибрано 1159 дублікатів за group_id (Україна)


(10099, 93)

In [20]:
before = len(df_kyiv)
df_encoded_kyiv = df_kyiv.drop_duplicates(subset="group_id", keep="first").copy()
print(f"Прибрано {before - len(df_encoded_kyiv)} дублікатів за group_id (Київ)")

df_encoded_kyiv = df_encoded_kyiv.drop(columns=["url", "group_id", "geo_region", "text"])

df_encoded_kyiv = df_encoded_kyiv.drop(columns=["city"])
categorical_cols_kyiv = ["house_type", "wall_type", "heating", "kitchen_type", "hot_water_type", "district"]
df_encoded_kyiv = pd.get_dummies(df_encoded_kyiv, columns=categorical_cols_kyiv, drop_first=True)
df_encoded_kyiv.to_csv("Kyiv_flats_ML.csv", index=False)
df_encoded_kyiv.shape

Прибрано 504 дублікатів за group_id (Київ)


(9970, 78)